## 2026 EY AI & Data Challenge - TerraClimate Data Extraction Notebook

This notebooks demonstrates how to access the TerraClimate dataset. TerraClimate is a dataset of monthly climate and climatic water balance for global terrestrial surfaces from 1958 to the present. These data provide important inputs for ecological and hydrological studies at global scales that require high spatial resolution and time-varying data. All data have monthly temporal resolution and a ~4-km (1/24th degree) spatial resolution. This dataset is provided in Zarr format. 

For more information, visit: [terraclimate- overview](https://planetarycomputer.microsoft.com/dataset/terraclimate#overview) 

## Load In Dependencies
The following code installs the required Python libraries (found in the requirements.txt file) in the Snowflake environment to allow successful execution of the remaining notebook code. After running this code for the first time, it is required to “restart” the kernal so the Python libraries are available in the environment. This is done by selecting the “Connected” menu above the notebook (next to “Run all”) and selecting the “restart kernal” link. Subsequent runs of the notebook do not require this “restart” process.

In [1]:
!pip install uv
!uv pip install  -r requirements.txt 


[notice] A new release of pip available: 22.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip
Audited 20 packages in 22ms


In [1]:
import warnings
warnings.filterwarnings("ignore")

# Data manipulation and analysis
import numpy as np
import pandas as pd

# Multi-dimensional arrays and datasets (e.g., NetCDF, Zarr)
import xarray as xr

from scipy.spatial import cKDTree

# Planetary Computer tools for STAC API access and authentication
import pystac_client
import planetary_computer as pc

from datetime import date
from tqdm import tqdm
import time
import os
import tempfile

## Extracting TerraClimate Data Using API Calls

The API-based method allows us to efficiently access **TerraClimate** data for specific regions and time periods through the [Microsoft Planetary Computer](https://planetarycomputer.microsoft.com/), ensuring scalability and reproducibility of the process.

Through the API, we can extract climate variables such as **Potential Evapotranspiration (PET)**, which represents the atmospheric demand for water. This variable provides important insights into surface moisture balance and helps improve the accuracy of water quality modeling.

This approach ensures consistent, automated retrieval of high-resolution climate data that can be easily integrated with satellite-derived features for comprehensive environmental and hydrological analysis.

### Loading and Mapping TerraClimate Data

This section demonstrates how **TerraClimate climate variables**, such as **Potential Evapotranspiration (PET)**, are loaded and mapped to sampling locations.

- The **load_terraclimate_dataset** function opens the TerraClimate Zarr/NetCDF dataset from the Microsoft Planetary Computer, handling storage options automatically.
- The **filterg** function filters the dataset for the desired time range (2011–2015) and the spatial extent corresponding to the study region. The resulting data is converted into a pandas DataFrame with standardized column names.
- The **assign_nearest_climate** function maps each sampling location to its **nearest TerraClimate grid point** using a KD-tree and assigns the climate variable values corresponding to the closest timestamp.

This workflow ensures efficient, reproducible retrieval of climate variables, while allowing participants to work with pre-extracted CSV files for faster benchmarking and analysis.

In [2]:
def load_terraclimate_dataset(sleep_sec=1):
    # Pause before opening remote dataset to avoid hammering the API
    time.sleep(sleep_sec)
    catalog = pystac_client.Client.open(
        "https://planetarycomputer.microsoft.com/api/stac/v1",
        modifier=pc.sign_inplace,
    )
    collection = catalog.get_collection("terraclimate")
    asset = collection.assets["zarr-abfs"]

    if "xarray:storage_options" in asset.extra_fields:
        ds = xr.open_zarr(
            asset.href,
            storage_options=asset.extra_fields["xarray:storage_options"],
            consolidated=True,
        )
    else:
        ds = xr.open_dataset(
            asset.href,
            **asset.extra_fields["xarray:open_kwargs"],
        )

    return ds

In [3]:
# --- Filtering function (kept identical) ---
def filterg(ds, var):
    ds_2011_2015 = ds[var].sel(time=slice("2011-01-01", "2015-12-31"))

    df_var_append = []
    for i in tqdm(range(len(ds_2011_2015.time))):
        df_var = ds_2011_2015.isel(time=i).to_dataframe().reset_index()
        df_var_filter = df_var[
            (df_var['lat'] > -35.18) & (df_var['lat'] < -21.72) &
            (df_var['lon'] > 14.97) & (df_var['lon'] < 32.79)
        ]
        df_var_append.append(df_var_filter)

    df_var_final = pd.concat(df_var_append, ignore_index=True)
    print(f"Filtering for {var} completed")

    df_var_final['time'] = df_var_final['time'].astype(str)

    # Column mapping
    col_mapping = {"lat": "Latitude", "lon": "Longitude", "time": "Sample Date"}
    df_var_final = df_var_final.rename(columns=col_mapping)

    return df_var_final

In [4]:
# --- Climate variable assignment function (unchanged logic) ---
def assign_nearest_climate(sa_df, climate_df, var_name):
    """
    Map nearest climate variable values to a new DataFrame 
    containing only the specified variable column.
    """
    sa_coords = np.radians(sa_df[['Latitude', 'Longitude']].values)
    climate_coords = np.radians(climate_df[['Latitude', 'Longitude']].values)

    tree = cKDTree(climate_coords)
    dist, idx = tree.query(sa_coords, k=1)

    nearest_points = climate_df.iloc[idx].reset_index(drop=True)

    sa_df = sa_df.reset_index(drop=True)
    sa_df[['nearest_lat', 'nearest_lon']] = nearest_points[['Latitude', 'Longitude']]

    sa_df['Sample Date'] = pd.to_datetime(sa_df['Sample Date'], dayfirst=True, errors='coerce')
    climate_df['Sample Date'] = pd.to_datetime(climate_df['Sample Date'], dayfirst=True, errors='coerce')

    climate_values = []

    for i in tqdm(range(len(sa_df)), desc=f"Mapping {var_name.upper()} values"):
        sample_date = sa_df.loc[i, 'Sample Date']
        nearest_lat = sa_df.loc[i, 'nearest_lat']
        nearest_lon = sa_df.loc[i, 'nearest_lon']

        subset = climate_df[
            (climate_df['Latitude'] == nearest_lat) &
            (climate_df['Longitude'] == nearest_lon)
        ]

        if subset.empty:
            climate_values.append(np.nan)
            continue

        nearest_idx = (subset['Sample Date'] - sample_date).abs().idxmin()
        climate_values.append(subset.loc[nearest_idx, var_name])

    output_df = pd.DataFrame({var_name: climate_values})

    
    return output_df

### Extracting features for the training dataset

In [14]:
Water_Quality_df = pd.read_csv("water_quality_new_training_dataset.csv")
display(Water_Quality_df.head(5))

,Latitude,Longitude,Sample Date,Total Alkalinity,Electrical Conductance,Dissolved Reactive Phosphorus
0,-34.096389,24.439167,01-02-2005,20.851,431.0,0.019
1,-34.096389,24.439167,02-08-2005,21.489,651.0,0.012
2,-34.096389,24.439167,02-09-2003,9.555,489.0,0.043
3,-34.096389,24.439167,02-12-2003,21.898,661.0,0.023
4,-34.096389,24.439167,02-12-2014,14.297,582.0,0.068


In [15]:
# remover datas que sao antes de 2011-01-01 e depois de 2015-12-31
Water_Quality_df['_date'] = pd.to_datetime(Water_Quality_df['Sample Date'], dayfirst=True, errors='coerce')
Water_Quality_df = Water_Quality_df[(Water_Quality_df['_date'] >= '2011-01-01') & (Water_Quality_df['_date'] <= '2015-12-31')]
Water_Quality_df = Water_Quality_df.drop(columns=['_date'])

In [16]:
Water_Quality_df.shape

(485, 6)

In [17]:

# ── Shared config (run once, used by all extraction cells below) ────────────
import os
import tempfile

VARS_TO_EXTRACT   = ['ppt', 'soil', 'def', 'q', 'aet']
chunksize         = 500
sleep_between     = 2
output_path       = "terraclimate_new_features_training_NP.csv"
tmp_path          = os.path.join(tempfile.gettempdir(), "terraclimate_new_features_training_NP.csv")
progress_path     = "terraclimate_train_progress.txt"
tmp_progress_path = os.path.join(tempfile.gettempdir(), "terraclimate_train_progress.txt")

# Partial results accumulator — persists across cells in the same session
if "train_var_results" not in dir():
    train_var_results = {}

print(f"✅ Config ready. {len(Water_Quality_df)} rows in training dataset.")
print(f"   Variables collected so far: {list(train_var_results.keys())}")

✅ Config ready. 485 rows in training dataset.
   Variables collected so far: []


In [18]:

# ── CELL A: ppt + soil ──────────────────────────────────────────────────────
for var in ['ppt', 'soil']:
    if var in train_var_results:
        print(f"⏭️  {var} already extracted, skipping.")
        continue
    print(f"\n🔄 Fresh token + extraction for: {var}")
    ds = load_terraclimate_dataset()
    tc_param = filterg(ds, var)
    result = assign_nearest_climate(Water_Quality_df, tc_param, var)
    train_var_results[var] = result[var].values
    print(f"✅ {var} done")


🔄 Fresh token + extraction for: ppt


100%|██████████| 60/60 [10:06<00:00, 10.11s/it]


Filtering for ppt completed


Mapping PPT values: 100%|██████████| 485/485 [00:09<00:00, 50.98it/s]


✅ ppt done

🔄 Fresh token + extraction for: soil


100%|██████████| 60/60 [07:30<00:00,  7.51s/it]


Filtering for soil completed


Mapping SOIL values: 100%|██████████| 485/485 [00:09<00:00, 50.82it/s]

✅ soil done


In [19]:

# ── CELL B: def + q ─────────────────────────────────────────────────────────
for var in ['def', 'q']:
    if var in train_var_results:
        print(f"⏭️  {var} already extracted, skipping.")
        continue
    print(f"\n🔄 Fresh token + extraction for: {var}")
    ds = load_terraclimate_dataset()
    tc_param = filterg(ds, var)
    result = assign_nearest_climate(Water_Quality_df, tc_param, var)
    train_var_results[var] = result[var].values
    print(f"✅ {var} done")


🔄 Fresh token + extraction for: def


100%|██████████| 60/60 [06:35<00:00,  6.60s/it]


Filtering for def completed


Mapping DEF values: 100%|██████████| 485/485 [00:08<00:00, 53.91it/s]


✅ def done

🔄 Fresh token + extraction for: q


100%|██████████| 60/60 [09:04<00:00,  9.07s/it]


Filtering for q completed


Mapping Q values: 100%|██████████| 485/485 [00:09<00:00, 53.54it/s]

✅ q done


In [21]:

# ── CELL C: aet ─────────────────────────────────────────────────────────────
for var in ['aet']:
    if var in train_var_results:
        print(f"⏭️  {var} already extracted, skipping.")
        continue
    print(f"\n🔄 Fresh token + extraction for: {var}")
    ds = load_terraclimate_dataset()
    tc_param = filterg(ds, var)
    result = assign_nearest_climate(Water_Quality_df, tc_param, var)
    train_var_results[var] = result[var].values
    print(f"✅ {var} done")


🔄 Fresh token + extraction for: aet


100%|██████████| 60/60 [06:32<00:00,  6.55s/it]


Filtering for aet completed


Mapping AET values: 100%|██████████| 485/485 [00:09<00:00, 51.50it/s]

✅ aet done


In [22]:

# ── CELL D: Merge all variables → Terraclimate_training_df ──────────────────
missing = [v for v in VARS_TO_EXTRACT if v not in train_var_results]
if missing:
    print(f"⚠️  Still missing variables: {missing}. Run the cells above first.")
else:
    Terraclimate_training_df = pd.DataFrame({v: train_var_results[v] for v in VARS_TO_EXTRACT})
    Terraclimate_training_df['Latitude']    = Water_Quality_df['Latitude'].values
    Terraclimate_training_df['Longitude']   = Water_Quality_df['Longitude'].values
    Terraclimate_training_df['Sample Date'] = Water_Quality_df['Sample Date'].values

    Terraclimate_training_df.to_csv(output_path, index=False)
    try:
        Terraclimate_training_df.to_csv(tmp_path, index=False)
        print(f"✅ Backup saved to {tmp_path}")
    except Exception as e:
        print(f"⚠️  Could not save backup to temp: {e}")
    print(f"🎉 Done! {len(Terraclimate_training_df)} rows saved to {output_path}")
    display(Terraclimate_training_df.head())

✅ Backup saved to C:\Users\celso\AppData\Local\Temp\terraclimate_new_features_training_NP.csv
🎉 Done! 485 rows saved to terraclimate_new_features_training_NP.csv


,ppt,soil,def,q,aet,Latitude,Longitude,Sample Date
0,34.9,8.100000,115.400002,1.7,34.500000,-34.096389,24.439167,02-12-2014
1,12.6,13.600000,134.400009,0.6,16.300001,-34.096389,24.439167,04-02-2015
2,12.6,13.600000,134.400009,0.6,16.300001,-34.096389,24.439167,04-03-2015
3,86.2,11.400001,45.299999,4.3,84.700005,-34.096389,24.439167,04-05-2011
4,54.3,12.600000,86.900002,2.7,55.200001,-34.096389,24.439167,04-12-2013


In [23]:
# Preview File
display(Terraclimate_training_df.head())

,ppt,soil,def,q,aet,Latitude,Longitude,Sample Date
0,34.9,8.100000,115.400002,1.7,34.500000,-34.096389,24.439167,02-12-2014
1,12.6,13.600000,134.400009,0.6,16.300001,-34.096389,24.439167,04-02-2015
2,12.6,13.600000,134.400009,0.6,16.300001,-34.096389,24.439167,04-03-2015
3,86.2,11.400001,45.299999,4.3,84.700005,-34.096389,24.439167,04-05-2011
4,54.3,12.600000,86.900002,2.7,55.200001,-34.096389,24.439167,04-12-2013


In [24]:
Terraclimate_training_df.to_csv("terraclimate_new_features_training(NT).csv", index=False)

**Note:** If you're using your own workspace, remember to replace "EY-AI-and-Data-Challenge" with your workspace name in the file path.

### Extracting features for the validation dataset

In [25]:
Validation_df=pd.read_csv('submission_template.csv')
display(Validation_df.head())

,Latitude,Longitude,Sample Date,Total Alkalinity,Electrical Conductance,Dissolved Reactive Phosphorus
0,-32.043333,27.822778,01-09-2014,NaN,NaN,NaN
1,-33.329167,26.077500,16-09-2015,NaN,NaN,NaN
2,-32.991639,27.640028,07-05-2015,NaN,NaN,NaN
3,-34.096389,24.439167,07-02-2012,NaN,NaN,NaN
4,-32.000556,28.581667,01-10-2014,NaN,NaN,NaN


In [26]:
Validation_df.shape

(200, 6)

In [27]:

# ── Shared config (run once, used by all validation extraction cells below) ─
import os
import tempfile

VARS_TO_EXTRACT    = ['ppt', 'soil', 'def', 'q', 'aet']
val_output_path    = "terraclimate_new_features_validation.csv"
val_tmp_path       = os.path.join(tempfile.gettempdir(), "terraclimate_new_features_validation.csv")

Validation_df = pd.read_csv('submission_template.csv')

# Partial results accumulator — persists across cells in the same session
if "val_var_results" not in dir():
    val_var_results = {}

print(f"✅ Config ready. {len(Validation_df)} rows in validation dataset.")
print(f"   Variables collected so far: {list(val_var_results.keys())}")

✅ Config ready. 200 rows in validation dataset.
   Variables collected so far: []


In [28]:

# ── CELL A: ppt + soil ──────────────────────────────────────────────────────
for var in ['ppt', 'soil']:
    if var in val_var_results:
        print(f"⏭️  {var} already extracted, skipping.")
        continue
    print(f"\n🔄 Fresh token + extraction for: {var}")
    ds = load_terraclimate_dataset()
    tc_param = filterg(ds, var)
    result = assign_nearest_climate(Validation_df, tc_param, var)
    val_var_results[var] = result[var].values
    print(f"✅ {var} done")


🔄 Fresh token + extraction for: ppt


  8%|▊         | 5/60 [00:59<10:53, 11.88s/it]


KeyboardInterrupt: 

In [36]:

# ── CELL B: def + q ─────────────────────────────────────────────────────────
for var in ['def', 'q']:
    if var in val_var_results:
        print(f"⏭️  {var} already extracted, skipping.")
        continue
    print(f"\n🔄 Fresh token + extraction for: {var}")
    ds = load_terraclimate_dataset()
    tc_param = filterg(ds, var)
    result = assign_nearest_climate(Validation_df, tc_param, var)
    val_var_results[var] = result[var].values
    print(f"✅ {var} done")


🔄 Fresh token + extraction for: def


100%|██████████| 60/60 [06:48<00:00,  6.80s/it]


Filtering for def completed


Mapping DEF values: 100%|██████████| 200/200 [00:03<00:00, 50.79it/s]


✅ def done

🔄 Fresh token + extraction for: q


100%|██████████| 60/60 [07:33<00:00,  7.56s/it]


Filtering for q completed


Mapping Q values: 100%|██████████| 200/200 [00:03<00:00, 59.41it/s]

✅ q done


In [37]:

# ── CELL C: aet ─────────────────────────────────────────────────────────────
for var in ['aet']:
    if var in val_var_results:
        print(f"⏭️  {var} already extracted, skipping.")
        continue
    print(f"\n🔄 Fresh token + extraction for: {var}")
    ds = load_terraclimate_dataset()
    tc_param = filterg(ds, var)
    result = assign_nearest_climate(Validation_df, tc_param, var)
    val_var_results[var] = result[var].values
    print(f"✅ {var} done")


🔄 Fresh token + extraction for: aet


100%|██████████| 60/60 [06:17<00:00,  6.28s/it]


Filtering for aet completed


Mapping AET values: 100%|██████████| 200/200 [00:03<00:00, 58.34it/s]

✅ aet done


In [38]:

# ── CELL D: Merge all variables → Terraclimate_validation_df ────────────────
missing = [v for v in VARS_TO_EXTRACT if v not in val_var_results]
if missing:
    print(f"⚠️  Still missing variables: {missing}. Run the cells above first.")
else:
    Terraclimate_validation_df = pd.DataFrame({v: val_var_results[v] for v in VARS_TO_EXTRACT})
    Terraclimate_validation_df['Latitude']    = Validation_df['Latitude'].values
    Terraclimate_validation_df['Longitude']   = Validation_df['Longitude'].values
    Terraclimate_validation_df['Sample Date'] = Validation_df['Sample Date'].values
    Terraclimate_validation_df = Terraclimate_validation_df[['Latitude', 'Longitude', 'Sample Date', 'ppt', 'soil', 'def', 'aet', 'q']]

    Terraclimate_validation_df.to_csv(val_output_path, index=False)
    try:
        Terraclimate_validation_df.to_csv(val_tmp_path, index=False)
        print(f"✅ Backup saved to {val_tmp_path}")
    except Exception as e:
        print(f"⚠️  Could not save backup to temp: {e}")
    print(f"🎉 Done! {len(Terraclimate_validation_df)} rows saved to {val_output_path}")
    display(Terraclimate_validation_df.head())

✅ Backup saved to C:\Users\celso\AppData\Local\Temp\terraclimate_new_features_validation.csv
🎉 Done! 200 rows saved to terraclimate_new_features_validation.csv


,Latitude,Longitude,Sample Date,ppt,soil,def,aet,q
0,-32.043333,27.822778,01-09-2014,25.6,4.8,137.100006,24.800001,1.3
1,-33.329167,26.077500,16-09-2015,11.6,2.7,166.400009,11.200000,0.6
2,-32.991639,27.640028,07-05-2015,21.5,12.0,136.300003,22.100000,1.1
3,-34.096389,24.439167,07-02-2012,28.2,16.4,95.900002,34.100002,1.4
4,-32.000556,28.581667,01-10-2014,27.2,5.1,126.099998,26.400000,1.4


In [39]:
Terraclimate_validation_df['Latitude'] = Validation_df['Latitude']
Terraclimate_validation_df['Longitude'] = Validation_df['Longitude']
Terraclimate_validation_df['Sample Date'] = Validation_df['Sample Date']
Terraclimate_validation_df = Terraclimate_validation_df[['Latitude', 'Longitude', 'Sample Date', 'ppt', 'soil', 'def', 'aet', 'q']]
Terraclimate_validation_df.to_csv('terraclimate_new_features_validation.csv', index=False)

In [40]:
# Preview File
display(Terraclimate_validation_df.head())

,Latitude,Longitude,Sample Date,ppt,soil,def,aet,q
0,-32.043333,27.822778,01-09-2014,25.6,4.8,137.100006,24.800001,1.3
1,-33.329167,26.077500,16-09-2015,11.6,2.7,166.400009,11.200000,0.6
2,-32.991639,27.640028,07-05-2015,21.5,12.0,136.300003,22.100000,1.1
3,-34.096389,24.439167,07-02-2012,28.2,16.4,95.900002,34.100002,1.4
4,-32.000556,28.581667,01-10-2014,27.2,5.1,126.099998,26.400000,1.4


In [41]:
Terraclimate_validation_df.to_csv("terraclimate_new_features_validation.csv", index=False)

**Note:** If you're using your own workspace, remember to replace "EY-AI-and-Data-Challenge" with your workspace name in the file path.